In [1]:
"""
Search for LPG points of sale (filling + reseller) in all capitals and second cities
of continental sub-Saharan Africa (including Madagascar).
Sources: OpenStreetMap (Overpass) + Google Places (multilingual).
Output: CSV, JSON, GeoPackage by city, by country and overall dataset.
"""

import requests
import time
import csv
import json
import os
import math
from datetime import datetime
from collections import Counter

# ======================== CONFIGURATION ========================
USE_GOOGLE = False
API_KEY = "LA_TUA_API_KEY_GOOGLE"          # <-- insert valid key

STEP = 0.2          # search grid degrees (approx. 22 km)
RADIUS = 50000      # radius in metres for Google Nearby Search

# Output root folder
OUTPUT_ROOT = "output_africa"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ======================== KEYWORDS ========================
# ----- RESELLER (reseller) -----
RESELLER_KEYWORDS = {
    'en': [
        "LPG refill station", "gas station with LPG", "LPG filling point",
        "LPG dealer", "gas cylinder exchange", "cooking gas", "LPG",
        "butane gas", "butane cylinder"
    ],
    'fr': [
        "station de remplissage de GPL", "station-service avec GPL",
        "point de remplissage de GPL", "revendeur de GPL",
        "échange de bouteilles de gaz", "gaz de cuisine", "GPL",
        "gaz butane", "bouteille de gaz butane"
    ],
    'pt': [
        "estação de recarga de GLP", "posto de gasolina com GLP",
        "ponto de enchimento de GLP", "revendedor de GLP",
        "troca de botijão de gás", "gás de cozinha", "GLP",
        "gás butano", "botija de gás butano"
    ],
    'sw': [
        "kituo cha kujaza LPG", "kituo cha mafuta chenye LPG",
        "sehemu ya kujaza LPG", "muuzaji wa LPG",
        "kubadilisha mitungi ya gesi", "gesi ya kupikia", "LPG",
        "gesi ya butane", "mtungi wa gesi ya butane"
    ],
    'ar': [
        "محطة تعبئة غاز البترول المسال", "محطة وقود مع غاز البترول المسال",
        "نقطة تعبئة غاز البترول المسال", "تاجر غاز البترول المسال",
        "استبدال اسطوانات الغاز", "غاز الطبخ", "غاز البترول المسال",
        "غاز البيوتان", "اسطوانة غاز البيوتان"
    ]
}

# ----- FILLING PLANTS (filling) -----
FILLING_KEYWORDS = {
    'en': [
        "LPG cylinder filling plant", "LPG bottling plant",
        "gas bottling plant", "LPG filling plant",
        "cylinder filling station", "LPG bottling station"
    ],
    'fr': [
        "usine de remplissage de bouteilles de GPL", "usine d'embouteillage de GPL",
        "usine d'embouteillage de gaz", "usine de remplissage de GPL",
        "station de remplissage de bouteilles", "station d'embouteillage de GPL"
    ],
    'pt': [
        "fábrica de enchimento de botijas de GLP", "fábrica de engarrafamento de GLP",
        "fábrica de engarrafamento de gás", "fábrica de enchimento de GLP",
        "estação de enchimento de botijas", "estação de engarrafamento de GLP"
    ],
    'sw': [
        "kiwanda cha kujaza mitungi ya LPG",
        "kiwanda cha kujaza LPG",
        "kiwanda cha kujaza gesi",
        "kiwanda cha kusindika LPG",
        "kituo cha kujaza mitungi",
        "kituo cha kujaza LPG"
    ],
    'ar': [
        "مصنع تعبئة اسطوانات غاز البترول المسال", "مصنع تعبئة غاز البترول المسال",
        "مصنع تعبئة الغاز", "محطة تعبئة اسطوانات الغاز",
        "محطة تعبئة غاز البترول المسال", "محطة تعبئة الغاز"
    ]
}

# ======================== LANGUAGE MAP PER COUNTRY ========================
COUNTRY_LANGUAGES = {
    'angola':             ['en', 'pt'],
    'benin':              ['en', 'fr'],
    'botswana':           ['en'],
    'burkina_faso':       ['en', 'fr'],
    'burundi':            ['en', 'fr', 'sw'],
    'cameroon':           ['en', 'fr'],
    'central_african_republic': ['en', 'fr'],
    'chad':               ['en', 'fr', 'ar'],
    'congo':              ['en', 'fr'],
    'congo_drc':          ['en', 'fr', 'sw'],
    'cote_divoire':       ['en', 'fr'],
    'djibouti':           ['en', 'fr', 'ar'],
    'equatorial_guinea':  ['en', 'pt', 'fr'],
    'eritrea':            ['en', 'ar'],
    'eswatini':           ['en'],
    'ethiopia':           ['en'],
    'gabon':              ['en', 'fr'],
    'gambia':             ['en'],
    'ghana':              ['en'],
    'guinea':             ['en', 'fr'],
    'guinea_bissau':      ['en', 'pt'],
    'kenya':              ['en', 'sw'],
    'lesotho':            ['en'],
    'liberia':            ['en'],
    'madagascar':         ['en', 'fr'],
    'malawi':             ['en'],
    'mali':               ['en', 'fr'],
    'mauritania':         ['en', 'fr', 'ar'],
    'mozambique':         ['en', 'pt'],
    'namibia':            ['en'],
    'niger':              ['en', 'fr'],
    'nigeria':            ['en'],
    'rwanda':             ['en', 'fr', 'sw'],
    'senegal':            ['en', 'fr'],
    'sierra_leone':       ['en'],
    'somalia':            ['en', 'ar'],
    'south_africa':       ['en'],
    'south_sudan':        ['en', 'ar', 'sw'],
    'sudan':              ['en', 'ar'],
    'tanzania':           ['en', 'sw'],
    'togo':               ['en', 'fr'],
    'uganda':             ['en', 'sw'],
    'zambia':             ['en'],
    'zimbabwe':           ['en']
}

# ======================== CITY LIST ========================
# Central coordinates for capital and second city; bbox automatic ±0.1°
CITIES_TO_PROCESS = [
    # Angola
    {'city': 'luanda',      'country': 'angola',        'lat': -8.838, 'lon': 13.234},
    {'city': 'huambo',      'country': 'angola',        'lat':-12.766, 'lon': 15.739},
    # Benin
    {'city': 'porto_novo',  'country': 'benin',         'lat': 6.483,  'lon': 2.616},
    {'city': 'cotonou',     'country': 'benin',         'lat': 6.370,  'lon': 2.428},
    # Botswana
    {'city': 'gaborone',    'country': 'botswana',      'lat':-24.628, 'lon': 25.923},
    {'city': 'francistown', 'country': 'botswana',      'lat':-21.171, 'lon': 27.514},
    # Burkina Faso
    {'city': 'ouagadougou', 'country': 'burkina_faso',  'lat': 12.371, 'lon': -1.528},
    {'city': 'bobo_dioulasso','country': 'burkina_faso','lat': 11.177, 'lon': -4.298},
    # Burundi
    {'city': 'gitega',      'country': 'burundi',       'lat': -3.427, 'lon': 29.925},
    {'city': 'bujumbura',   'country': 'burundi',       'lat': -3.382, 'lon': 29.360},
    # Camerun
    {'city': 'yaounde',     'country': 'cameroon',      'lat': 3.848,  'lon': 11.496},
    {'city': 'douala',      'country': 'cameroon',      'lat': 4.051,  'lon': 9.768},
    # Rep. Centrafricana
    {'city': 'bangui',      'country': 'central_african_republic','lat': 4.362, 'lon': 18.583},
    {'city': 'bimbo',       'country': 'central_african_republic','lat': 4.332, 'lon': 18.554},
    # Ciad
    {'city': 'n_djamena',   'country': 'chad',          'lat': 12.113, 'lon': 15.049},
    {'city': 'moundou',     'country': 'chad',          'lat': 8.567,  'lon': 16.083},
    # Congo
    {'city': 'brazzaville', 'country': 'congo',         'lat': -4.267, 'lon': 15.283},
    {'city': 'pointe_noire','country': 'congo',         'lat': -4.776, 'lon': 11.864},
    # Congo DRC
    {'city': 'kinshasa',    'country': 'congo_drc',     'lat': -4.326, 'lon': 15.309},
    {'city': 'lubumbashi',  'country': 'congo_drc',     'lat':-11.688, 'lon': 27.484},
    # Costa d'Avorio
    {'city': 'yamoussoukro','country': 'cote_divoire',  'lat': 6.828,  'lon': -5.277},
    {'city': 'abidjan',     'country': 'cote_divoire',  'lat': 5.345,  'lon': -4.022},
    # Gibuti
    {'city': 'djibouti_city','country': 'djibouti',     'lat': 11.589, 'lon': 43.148},
    {'city': 'ali_sabieh',  'country': 'djibouti',      'lat': 11.156, 'lon': 42.708},
    # Guinea Equatoriale
    {'city': 'malabo',      'country': 'equatorial_guinea','lat': 3.750, 'lon': 8.783},
    {'city': 'bata',        'country': 'equatorial_guinea','lat': 1.867, 'lon': 9.767},
    # Eritrea
    {'city': 'asmara',      'country': 'eritrea',       'lat': 15.328, 'lon': 38.932},
    {'city': 'massawa', 'country': 'eritrea', 'lat': 15.609, 'lon': 39.450},
    # eSwatini
    {'city': 'mbabane',     'country': 'eswatini',      'lat':-26.320, 'lon': 31.144},
    {'city': 'manzini',     'country': 'eswatini',      'lat':-26.503, 'lon': 31.372},
    # Etiopia
    {'city': 'addis_ababa', 'country': 'ethiopia',      'lat': 9.024,  'lon': 38.757},
    {'city': 'dire_dawa',   'country': 'ethiopia',      'lat': 9.593,  'lon': 41.862},
    # Gabon
    {'city': 'libreville',  'country': 'gabon',         'lat': 0.390,  'lon': 9.454},
    {'city': 'port_gentil', 'country': 'gabon',         'lat': -0.719, 'lon': 8.781},
    # Gambia
    {'city': 'banjul',      'country': 'gambia',        'lat': 13.453, 'lon':-16.579},
    {'city': 'serekunda',   'country': 'gambia',        'lat': 13.423, 'lon':-16.678},
    # Ghana
    {'city': 'accra',       'country': 'ghana',         'lat': 5.556,  'lon': -0.201},
    {'city': 'kumasi',      'country': 'ghana',         'lat': 6.688,  'lon': -1.624},
    # Guinea
    {'city': 'conakry',     'country': 'guinea',        'lat': 9.509,  'lon':-13.712},
    {'city': 'nzerekore',   'country': 'guinea',        'lat': 7.758,  'lon': -8.818},
    # Guinea-Bissau
    {'city': 'bissau',      'country': 'guinea_bissau', 'lat': 11.860, 'lon':-15.598},
    {'city': 'bafata',      'country': 'guinea_bissau', 'lat': 12.167, 'lon':-14.663},
    # Kenya
    {'city': 'nairobi',     'country': 'kenya',         'lat': -1.286, 'lon': 36.817},
    {'city': 'mombasa',     'country': 'kenya',         'lat': -4.043, 'lon': 39.668},
    # Lesotho
    {'city': 'maseru',      'country': 'lesotho',       'lat':-29.316, 'lon': 27.483},
    {'city': 'teyateyaneng','country': 'lesotho',       'lat':-29.150, 'lon': 27.733},
    # Liberia
    {'city': 'monrovia',    'country': 'liberia',       'lat': 6.301,  'lon':-10.797},
    {'city': 'ganta',       'country': 'liberia',       'lat': 7.234,  'lon': -8.991},
    # Madagascar
    {'city': 'antananarivo','country': 'madagascar',    'lat':-18.914, 'lon': 47.536},
    {'city': 'toamasina',   'country': 'madagascar',    'lat':-18.149, 'lon': 49.402},
    # Malawi
    {'city': 'lilongwe',    'country': 'malawi',        'lat':-13.983, 'lon': 33.783},
    {'city': 'blantyre',    'country': 'malawi',        'lat':-15.786, 'lon': 35.008},
    # Mali
    {'city': 'bamako',      'country': 'mali',          'lat': 12.639, 'lon': -8.002},
    {'city': 'segou',       'country': 'mali',          'lat': 13.431, 'lon': -6.266},
    # Mauritania
    {'city': 'nouakchott',  'country': 'mauritania',    'lat': 18.086, 'lon':-15.975},
    {'city': 'nouadhibou',  'country': 'mauritania',    'lat': 20.933, 'lon':-17.033},
    # Mozambico
    {'city': 'maputo',      'country': 'mozambique',    'lat':-25.969, 'lon': 32.573},
    {'city': 'beira',       'country': 'mozambique',    'lat':-19.840, 'lon': 34.838},
    # Namibia
    {'city': 'windhoek',    'country': 'namibia',       'lat':-22.561, 'lon': 17.086},
    {'city': 'walvis_bay',  'country': 'namibia',       'lat':-22.957, 'lon': 14.505},
    # Niger
    {'city': 'niamey',      'country': 'niger',         'lat': 13.512, 'lon': 2.112},
    {'city': 'zinder',      'country': 'niger',         'lat': 13.807, 'lon': 8.989},
    # Nigeria
    {'city': 'abuja',       'country': 'nigeria',       'lat': 9.057,  'lon': 7.489},
    {'city': 'lagos',       'country': 'nigeria',       'lat': 6.455,  'lon': 3.394},
    # Ruanda
    {'city': 'kigali',      'country': 'rwanda',        'lat': -1.950, 'lon': 30.060},
    {'city': 'gisenyi',     'country': 'rwanda',        'lat': -1.702, 'lon': 29.260},
    # Senegal
    {'city': 'dakar',       'country': 'senegal',       'lat': 14.694, 'lon':-17.446},
    {'city': 'touba',       'country': 'senegal',       'lat': 14.861, 'lon':-15.881},
    # Sierra Leone
    {'city': 'freetown',    'country': 'sierra_leone',  'lat': 8.484,  'lon':-13.234},
    {'city': 'bo',          'country': 'sierra_leone',  'lat': 7.962,  'lon':-11.740},
    # Somalia
    {'city': 'mogadishu',   'country': 'somalia',       'lat': 2.037,  'lon': 45.343},
    {'city': 'hargeisa',    'country': 'somalia',       'lat': 9.562,  'lon': 44.077},
    # Sudafrica
    {'city': 'pretoria',    'country': 'south_africa',  'lat':-25.746, 'lon': 28.188},
    {'city': 'johannesburg','country': 'south_africa',  'lat':-26.204, 'lon': 28.047},
    # Sud Sudan
    {'city': 'juba',        'country': 'south_sudan',   'lat': 4.860,  'lon': 31.591},
    {'city': 'wau',         'country': 'south_sudan',   'lat': 7.702,  'lon': 27.997},
    # Sudan
    {'city': 'khartoum',    'country': 'sudan',         'lat': 15.552, 'lon': 32.560},
    {'city': 'omdurman',    'country': 'sudan',         'lat': 15.647, 'lon': 32.481},
    # Tanzania
    {'city': 'dodoma',      'country': 'tanzania',      'lat': -6.173, 'lon': 35.742},
    {'city': 'dar_es_salaam','country': 'tanzania',     'lat': -6.792, 'lon': 39.208},
    # Togo
    {'city': 'lome',        'country': 'togo',          'lat': 6.128,  'lon': 1.225},
    {'city': 'sokode',      'country': 'togo',          'lat': 8.991,  'lon': 1.145},
    # Uganda
    {'city': 'kampala',     'country': 'uganda',        'lat': 0.313,  'lon': 32.581},
    {'city': 'gulu',        'country': 'uganda',        'lat': 2.772,  'lon': 32.299},
    # Zambia
    {'city': 'lusaka',      'country': 'zambia',        'lat':-15.417, 'lon': 28.283},
    {'city': 'kitwe',       'country': 'zambia',        'lat':-12.823, 'lon': 28.202},
    # Zimbabwe
    {'city': 'harare',      'country': 'zimbabwe',      'lat':-17.822, 'lon': 31.050},
    {'city': 'bulawayo',    'country': 'zimbabwe',      'lat':-20.171, 'lon': 28.587}
]

# ======================== OSM AND GOOGLE FUNCTIONS (logic unchanged) ========================
def osm_search(lat, lon, radius=50000):
    delta = radius / 111000.0
    min_lat = lat - delta
    max_lat = lat + delta
    min_lon = lon - delta
    max_lon = lon + delta

    query = f"""
    [out:json];
    (
      node["amenity"="fuel"]["fuel:lpg"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="fuel"]["fuel:lpg"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["amenity"="fuel"]["fuel:GPL"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="fuel"]["fuel:GPL"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["industrial"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["man_made"="storage_tank"]["content"~"lpg|gpl|glp"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    url = "https://overpass-api.de/api/interpreter"
    headers = {
        'User-Agent': 'LPG-Research-Script/1.0 (contact: your-email@example.com)',
        'Accept': 'application/json'
    }
    try:
        response = requests.post(url, data={'data': query}, headers=headers, timeout=30)
        if response.status_code != 200:
            print(f"  ⚠️ OSM HTTP error {response.status_code}: {response.text[:100]}")
            return []
        data = response.json()
    except Exception as e:
        print(f"  ⚠️ OSM request error: {e}")
        return []

    results = []
    for elem in data.get('elements', []):
        if elem['type'] == 'node':
            lat_place = elem.get('lat')
            lon_place = elem.get('lon')
        else:
            center = elem.get('center', {})
            lat_place = center.get('lat')
            lon_place = center.get('lon')
        if lat_place is None or lon_place is None:
            continue
        tags = elem.get('tags', {})
        if 'amenity' in tags and tags['amenity'] == 'fuel':
            ptype = 'fuel_station'
        elif 'shop' in tags:
            ptype = tags['shop']
        else:
            ptype = 'lpg_related'

        if (tags.get('industrial') == 'gas' or
            (tags.get('man_made') == 'storage_tank' and tags.get('content','').lower() in ['lpg','gpl','glp'])):
            category = 'filling'
        else:
            category = 'reseller'

        name = tags.get('name', '')
        results.append({
            'place_id': f"osm_{elem['id']}",
            'name': name,
            'address': '',
            'lat': lat_place,
            'lng': lon_place,
            'types': ptype,
            'keyword': 'osm',
            'search_lat': lat,
            'search_lon': lon,
            'rating': None,
            'user_ratings_total': 0,
            'source': 'osm',
            'language': '',
            'category': category
        })
    return results

def google_nearby_search(lat, lon, keyword, lang, category):
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    results = []
    params = {
        'location': f"{lat},{lon}",
        'radius': RADIUS,
        'keyword': keyword,
        'language': lang,
        'key': API_KEY
    }
    while True:
        resp = requests.get(url, params=params)
        data = resp.json()
        if data['status'] not in ['OK', 'ZERO_RESULTS']:
            error_msg = data.get('error_message', '')
            print(f"  ⚠️ Google status {data['status']} for {keyword}_{lang}: {error_msg}")
            break
        for place in data.get('results', []):
            results.append({
                'place_id': place['place_id'],
                'name': place.get('name', ''),
                'address': place.get('vicinity', ''),
                'lat': place['geometry']['location']['lat'],
                'lng': place['geometry']['location']['lng'],
                'types': ', '.join(place.get('types', [])),
                'keyword': f"{keyword}_{lang}",
                'search_lat': lat,
                'search_lon': lon,
                'rating': place.get('rating'),
                'user_ratings_total': place.get('user_ratings_total', 0),
                'source': 'google',
                'language': lang,
                'category': category
            })
        if 'next_page_token' in data:
            params['pagetoken'] = data['next_page_token']
            time.sleep(2)
        else:
            break
    return results

# ======================== SUPPORT FUNCTIONS ========================
def generate_grid(bbox, step):
    points = []
    lat = bbox['min_lat']
    while lat <= bbox['max_lat'] + 1e-9:
        lon = bbox['min_lon']
        while lon <= bbox['max_lon'] + 1e-9:
            points.append((round(lat, 4), round(lon, 4)))
            lon += step
        lat += step
    return points

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def compute_overlap(osm_points, google_points, max_dist=300):
    matched_osm = set()
    matched_google = set()
    pairs = []
    for g in google_points:
        for o in osm_points:
            d = haversine(o['lat'], o['lng'], g['lat'], g['lng'])
            if d <= max_dist:
                matched_osm.add(o['place_id'])
                matched_google.add(g['place_id'])
                pairs.append((o['place_id'], g['place_id'], d))
    return matched_osm, matched_google, pairs

def print_rating_stats(results, label="ALL"):
    if not results:
        print(f"\n--- Rating stats ({label}): no points ---")
        return
    for r in results:
        try:
            r['rating_num'] = float(r['rating']) if r.get('rating') is not None else None
        except:
            r['rating_num'] = None
        try:
            r['reviews_num'] = float(r.get('user_ratings_total', 0) or 0)
        except:
            r['reviews_num'] = 0.0

    with_rating = [r for r in results if r['rating_num'] is not None]
    print(f"--- {label} ---")
    print(f"  Total: {len(results)}  |  With rating: {len(with_rating)}")
    if with_rating:
        ratings = [r['rating_num'] for r in with_rating]
        avg_r = sum(ratings)/len(ratings)
        med_r = sorted(ratings)[len(ratings)//2]
        print(f"  Avg rating: {avg_r:.2f}  |  Median: {med_r}")
        bins = [(0,1),(1,2),(2,3),(3,4),(4,5)]
        for low,high in bins:
            cnt = sum(1 for r in with_rating if low <= r['rating_num'] < high or (high==5 and r['rating_num']==5))
            print(f"    {low}≤r<{high}: {cnt}")
    reviews_all = [r['reviews_num'] for r in results]
    avg_rev = sum(reviews_all)/len(reviews_all) if reviews_all else 0
    med_rev = sorted(reviews_all)[len(reviews_all)//2] if reviews_all else 0
    print(f"  Reviews: avg {avg_rev:.1f}  |  median {med_rev}")
    # Only one line for points with few reviews
    few_rev = sum(1 for r in results if r['reviews_num'] < 2)
    print(f"  Points with <2 reviews: {few_rev}")

def print_google_stats(google_results):
    if not google_results:
        return
    print("  Google results by language:", Counter(r['language'] for r in google_results))
    kw_counter = Counter(r['keyword'].rsplit('_',1)[0] for r in google_results)
    print("  Google results by keyword:", kw_counter)

def save_city_files(results, filling, reseller, output_dir, city_prefix):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = os.path.join(output_dir, f"{city_prefix}_{timestamp}")
    fieldnames = ['place_id','name','address','lat','lng','types','keyword',
                  'search_lat','search_lon','rating','user_ratings_total','source','language','category']
    # CSV
    csv_file = f"{base}.csv"
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    print(f"✅ CSV: {csv_file}")
    # JSON
    json_file = f"{base}.json"
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON: {json_file}")
    # GeoPackage
    try:
        import geopandas as gpd
        import pandas as pd
        from shapely.geometry import Point
        keep_cols = ['place_id','name','address','types','keyword','search_lat','search_lon',
                     'rating','user_ratings_total','source','language','category']
        if filling:
            df = pd.DataFrame(filling)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_filling_3857.gpkg", driver='GPKG', layer='filling')
            print(f"✅ GeoPackage filling: {base}_filling_3857.gpkg")
        if reseller:
            df = pd.DataFrame(reseller)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_reseller_3857.gpkg", driver='GPKG', layer='reseller')
            print(f"✅ GeoPackage reseller: {base}_reseller_3857.gpkg")
    except ImportError:
        print("⚠️ GeoPandas not installed, skipping GeoPackage.")

def save_combined_files(all_cities_results, country_name, output_dir):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = os.path.join(output_dir, f"{country_name}_combined_{timestamp}")
    if not all_cities_results:
        print("  No points to save for the country.")
        return
    # CSV & JSON
    fieldnames = ['place_id','name','address','lat','lng','types','keyword',
                  'search_lat','search_lon','rating','user_ratings_total','source','language','category']
    with open(f"{base}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_cities_results)
    print(f"✅ Country CSV: {base}.csv")
    with open(f"{base}.json", 'w', encoding='utf-8') as f:
        json.dump(all_cities_results, f, indent=2, ensure_ascii=False)
    print(f"✅ Country JSON: {base}.json")
    # Separate GeoPackages
    filling = [r for r in all_cities_results if r['category'] == 'filling']
    reseller = [r for r in all_cities_results if r['category'] != 'filling']
    try:
        import geopandas as gpd
        import pandas as pd
        from shapely.geometry import Point
        keep = ['place_id','name','address','types','keyword','search_lat','search_lon',
                'rating','user_ratings_total','source','language','category']
        if filling:
            df = pd.DataFrame(filling)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_filling_3857.gpkg", driver='GPKG', layer='filling')
            print(f"✅ Country GeoPackage filling: {base}_filling_3857.gpkg")
        if reseller:
            df = pd.DataFrame(reseller)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_reseller_3857.gpkg", driver='GPKG', layer='reseller')
            print(f"✅ Country GeoPackage reseller: {base}_reseller_3857.gpkg")
    except ImportError:
        print("⚠️ GeoPandas not installed.")

# ======================== PROCESS FOR ONE CITY ========================
def process_city(city_config, country_name, country_langs, country_dir):
    city_name = city_config['city']
    lat_c, lon_c = city_config['lat'], city_config['lon']
    bbox = {
        'min_lat': round(lat_c - 0.1, 6),
        'max_lat': round(lat_c + 0.1, 6),
        'min_lon': round(lon_c - 0.1, 6),
        'max_lon': round(lon_c + 0.1, 6)
    }
    print(f"\n{'='*50}\nCity: {city_name.upper()}  (Country: {country_name})\n{'='*50}")
    grid = generate_grid(bbox, STEP)
    print(f"  Grid: {len(grid)} points")

    all_places = {}
    total_osm_calls = 0
    total_google_calls = 0

    for idx, (lat, lon) in enumerate(grid):
        if idx % 3 == 0:
            print(f"  Point {idx+1}/{len(grid)}: ({lat}, {lon})")

        # OSM
        try:
            osm_places = osm_search(lat, lon, RADIUS)
            total_osm_calls += 1
            for p in osm_places:
                if p['place_id'] not in all_places:
                    all_places[p['place_id']] = p
            time.sleep(1)
        except Exception as e:
            print(f"  ⚠️ OSM error: {e}")

        # Google
        if USE_GOOGLE and API_KEY != "LA_TUA_API_KEY_GOOGLE":
            # Filling
            for lang in country_langs:
                if lang not in FILLING_KEYWORDS: continue
                for kw in FILLING_KEYWORDS[lang]:
                    try:
                        places = google_nearby_search(lat, lon, kw, lang, 'filling')
                        total_google_calls += 1 + len(places)//60
                        for p in places:
                            pid = p['place_id']
                            if pid not in all_places:
                                all_places[pid] = p
                            else:
                                if all_places[pid].get('category') != 'filling':
                                    all_places[pid]['category'] = 'filling'
                    except Exception as e:
                        print(f"  ⚠️ Google filling {kw}_{lang}: {e}")
                    time.sleep(0.2)
            # Reseller
            for lang in country_langs:
                if lang not in RESELLER_KEYWORDS: continue
                for kw in RESELLER_KEYWORDS[lang]:
                    try:
                        places = google_nearby_search(lat, lon, kw, lang, 'reseller')
                        total_google_calls += 1 + len(places)//60
                        for p in places:
                            pid = p['place_id']
                            if pid not in all_places:
                                all_places[pid] = p
                    except Exception as e:
                        print(f"  ⚠️ Google reseller {kw}_{lang}: {e}")
                    time.sleep(0.2)

    results = list(all_places.values())
    osm_results = [r for r in results if r['source'] == 'osm']
    google_results = [r for r in results if r['source'] == 'google']
    filling_results = [r for r in results if r['category'] == 'filling']
    reseller_results = [r for r in results if r['category'] != 'filling']

    print(f"\n--- COLLECTION COMPLETED ---")
    print(f"  OSM calls: {total_osm_calls}  |  Google calls: {total_google_calls}")
    print(f"  Total places: {len(results)} (OSM: {len(osm_results)}, Google: {len(google_results)})")
    print(f"  Filling: {len(filling_results)}  |  Reseller: {len(reseller_results)}")

    # Google statistics
    if google_results:
        print_google_stats(google_results)

    # Overlap
    if osm_results and google_results:
        matched_osm, matched_google, pairs = compute_overlap(osm_results, google_results, 300)
        print(f"\n  Overlap OSM/Google (≤300 m): OSM matched: {len(matched_osm)}, Google matched: {len(matched_google)}, pairs: {len(pairs)}")

    # Rating stats
    print_rating_stats(results, "ALL POINTS")
    print_rating_stats(filling_results, "FILLING POINTS")
    print_rating_stats(reseller_results, "RESELLER POINTS")

    # Saving city files
    city_dir = os.path.join(country_dir, city_name)
    os.makedirs(city_dir, exist_ok=True)
    save_city_files(results, filling_results, reseller_results, city_dir, city_name)

    return results

# ======================== MAIN EXECUTION ========================
def main():
    all_countries_data = {}  # key: country, value: list of results from the two cities
    global_all_results = []  # all points from every city

    # Group cities by country
    country_cities = {}
    for cfg in CITIES_TO_PROCESS:
        country = cfg['country']
        country_cities.setdefault(country, []).append(cfg)

    for country, city_list in country_cities.items():
        print(f"\n\n{'#'*60}")
        print(f"  COUNTRY: {country.upper()}  ({len(city_list)} cities)")
        print(f"{'#'*60}")
        country_dir = os.path.join(OUTPUT_ROOT, country)
        os.makedirs(country_dir, exist_ok=True)

        langs = COUNTRY_LANGUAGES.get(country, ['en'])
        print(f"  Languages: {langs}")
        country_results = []

        for cfg in city_list:
            res = process_city(cfg, country, langs, country_dir)
            country_results.extend(res)
            time.sleep(2)  # pause between cities

        # Save national combined
        if country_results:
            save_combined_files(country_results, country, country_dir)
            # National statistics (brief)
            print(f"\n--- NATIONAL STATISTICS {country.upper()} ---")
            print(f"  Total points: {len(country_results)}")
            osm = [r for r in country_results if r['source']=='osm']
            gg = [r for r in country_results if r['source']=='google']
            print(f"  OSM: {len(osm)}  |  Google: {len(gg)}")
            filling = [r for r in country_results if r['category']=='filling']
            reseller = [r for r in country_results if r['category']=='reseller']
            print(f"  Filling: {len(filling)}  |  Reseller: {len(reseller)}")
        else:
            print(f"  No points for {country}")

        global_all_results.extend(country_results)

    # Final global save
    print(f"\n\n{'#'*60}")
    print("  SAVING GLOBAL DATASET (all cities)")
    print(f"{'#'*60}")
    global_dir = os.path.join(OUTPUT_ROOT, "_global")
    os.makedirs(global_dir, exist_ok=True)
    save_combined_files(global_all_results, "all_africa", global_dir)
    print("Processing completed.")

if __name__ == "__main__":
    if USE_GOOGLE and API_KEY == "LA_TUA_API_KEY_GOOGLE":
        print("⚠️ Insert a valid Google API key in the API_KEY variable.")
    main()



############################################################
  COUNTRY: ANGOLA  (2 cities)
############################################################
  Languages: ['en', 'pt']

City: LUANDA  (Country: angola)
  Grid: 4 points
  Point 1/4: (-8.938, 13.134)
  ⚠️ OSM HTTP error 504: <?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
    
  Point 4/4: (-8.738, 13.334)
  ⚠️ OSM HTTP error 429: <?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
    

--- COLLECTION COMPLETED ---
  OSM calls: 4  |  Google calls: 0
  Total places: 12 (OSM: 12, Google: 0)
  Filling: 0  |  Reseller: 12
--- ALL POINTS ---
  Total: 12  |  With rating: 0
  Reviews: avg 0.0  |  median 0.0
  Points with <2 reviews: 12

--- Rating stats (FILLING POINTS): no points ---
--- RESELLER POINTS ---
  Total: 12  |  With rating: 0
  Reviews: avg 0.0  |  median 0.0
  Points with <2 reviews: 12
✅ CSV: output_africa\angola\luanda\luanda_